# carbon-ets — quickstart

This notebook reproduces the core outputs of the Causal Systems report [Factories to carbon](https://causalsystems.co/research/factories-to-carbon) from free public data in about 60 seconds.

Everything runs on:
- **EEX primary-auction clearing prices** (2012 → today, no API key)
- **Eurostat industrial production** (via SDMX bulk endpoint)
- **European Commission MSR Communications** (verified TNAC values, hardcoded from the PDF source)

Open in Colab with the badge on the [GitHub README](https://github.com/causalsystems/carbon-ets).

In [ ]:
# Install (only needed in fresh Colab)
!pip install -q carbon-ets[yfinance]

## 1. The verified TNAC series

The single most useful reference in the toolkit. Every value has been extracted from the corresponding Commission Communication and hard-tested against the PDF source.

In [ ]:
from carbon_ets.tnac import get_reference_series, get_regime_context

df = get_reference_series()
df

In [ ]:
# What MSR regime does the most recent TNAC trigger?
latest = df['tnac'].iloc[-1]
ctx = get_regime_context(latest)
print(f"TNAC {df.index[-1].year}: {latest:,}")
print(f"Regime: {ctx['regime']}")
print(f"  {ctx['description']}")

## 2. TNAC history with MSR thresholds

The three horizontal lines are the policy-mechanism thresholds:
- **1.096 B** — above triggers 24% auction reduction
- **833 M** — between here and 1.096B triggers partial intake
- **400 M** — below triggers 100M release from MSR

In [ ]:
from carbon_ets.plots import plot_tnac_nowcast
plot_tnac_nowcast(df);

## 3. Historical EUA auction prices from EEX

Real-world compliance-buy prices, not an ETF proxy. Data starts January 2012.

In [ ]:
from carbon_ets.data import fetch_eua_prices
import matplotlib.pyplot as plt

eua = fetch_eua_prices(include_history=False)   # xlsx-only for speed in Colab

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(eua.index, eua['eua_eur_tco2'], color='#1a1a1a')
ax.set_title('EUA daily primary-auction clearing price')
ax.set_ylabel('EUR/tCO₂')
ax.grid(alpha=0.3)

## 4. Eurostat industrial production for EA19

The upstream driver in the demand-side chain. Seasonally adjusted, 2021 = 100.

In [ ]:
from carbon_ets.data import fetch_eurostat_ip

ip = fetch_eurostat_ip()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(ip.index, ip['ip_ea19'], color='#c0392b')
ax.set_title('EU-19 industrial production (SCA, index 2021=100)')
ax.axhline(100, color='grey', ls='--', alpha=0.5)
ax.grid(alpha=0.3)

## 5. End-to-end panel + feature engineering + V1 backtest

Reproduces the CS/RES/05 report's headline strategy in three function calls.

In [ ]:
from carbon_ets.data import build_full_panel
from carbon_ets.features import engineer
from carbon_ets.models import backtest_v1

panel = build_full_panel(start='2015-01-01')  # shorter window for speed in Colab
feats = engineer(panel)

stats, equity = backtest_v1(feats)
for k, v in stats.items():
    if isinstance(v, float):
        print(f"  {k:10s}: {v:+.4f}")
    else:
        print(f"  {k:10s}: {v}")

In [ ]:
from carbon_ets.plots import plot_equity_curve
plot_equity_curve(equity, title=f"EUA V1 causal-chain — Sharpe {stats['sharpe']:.2f}");

## What's next

- **Read the full report** — [causalsystems.co/research/factories-to-carbon](https://causalsystems.co/research/factories-to-carbon)
- **Explore the source code** — [github.com/causalsystems/carbon-ets](https://github.com/causalsystems/carbon-ets)
- **Contribute** — see `CONTRIBUTING.md` in the repo
- **Cite** — see the citation block in the README

Questions or suggestions: [hello@causalsystems.co](mailto:hello@causalsystems.co)